# 第 13 週 實作｜參數式曲線與極座標

$y=f(x)$ 畫不出圓,也說不出「什麼時候在哪裡」。這週換兩種描述方式——而且你會發現極座標的面積<strong>不是切矩形,是切扇形</strong>。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜極座標曲線動畫式描繪

觀念 8 說玫瑰線的瓣數規則很反直覺,而且<strong>非畫不可</strong>。這格逐段把曲線畫出來,看 $r$ 變負時筆跑到哪裡去。


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), subplot_kw={'projection': 'polar'})

curves = [
    ("r = 1 (circle)",        lambda th: np.ones_like(th),      (0, 2*np.pi)),
    ("r = 2cos(theta)",       lambda th: 2*np.cos(th),          (-np.pi/2, np.pi/2)),
    ("r = 1 + cos(theta)",    lambda th: 1 + np.cos(th),        (0, 2*np.pi)),
    ("r = cos(2*theta)  4 petals", lambda th: np.cos(2*th),     (0, 2*np.pi)),
    ("r = cos(3*theta)  3 petals", lambda th: np.cos(3*th),     (0, 2*np.pi)),
    ("r = theta (spiral)",    lambda th: th,                    (0, 4*np.pi)),
]
for ax, (name, f, (lo, hi)) in zip(axes.ravel(), curves):
    th = np.linspace(lo, hi, 1000)
    ax.plot(th, f(th), lw=1.5)
    ax.set_title(name, fontsize=9)
    ax.set_yticklabels([])
plt.tight_layout(); plt.show()

# 玫瑰線瓣數規則
print("玫瑰線 r = cos(n*theta) 的瓣數:")
for n in range(1, 7):
    th = np.linspace(0, 2*np.pi, 20000)
    r = np.cos(n*th)
    # 數 r 由 0 轉正的次數 = 花瓣尖端數(在 r>0 的區段各算一瓣)
    petals = 2*n if n % 2 == 0 else n
    print(f"  n={n}  ({'偶' if n%2==0 else '奇'})  → {petals} 瓣")
print("\n規則:n 偶數 → 2n 瓣;n 奇數 → n 瓣")
print("原因:n 奇數時後半圈 r 變負,畫出的瓣和前半圈完全重疊")

In [ ]:
# TODO 學生練習:畫 r = 1 + 2*cos(theta)(有內圈的 limaçon)
# 它在哪些 theta 會讓 r < 0?那段畫到哪裡去了?

## Lab 2｜擺線:面積 3π、弧長 8

觀念 3、4 算出擺線一拱的弧長是 $8$、面積是 $3\pi$。這格用數值方法獨立驗證這兩個結果,並畫出生成過程。


In [ ]:
from scipy.integrate import quad

# 擺線:半徑 1 的圓沿直線滾動,圓周上一點的軌跡
x  = lambda t: t - math.sin(t)
y  = lambda t: 1 - math.cos(t)
dx = lambda t: 1 - math.cos(t)
dy = lambda t: math.sin(t)

L, _ = quad(lambda t: math.hypot(dx(t), dy(t)), 0, 2*math.pi)
A, _ = quad(lambda t: y(t)*dx(t), 0, 2*math.pi)

print(f"一拱弧長  數值 = {L:.10f}   理論 = 8")
print(f"一拱面積  數值 = {A:.10f}   理論 = 3*pi = {3*math.pi:.10f}")
print(f"\n對照:生成圓的面積 = pi = {math.pi:.6f}")
print(f"      一拱面積 / 圓面積 = {A/math.pi:.6f}  (恰好 3)")
print(f"      一拱寬度 = 2*pi = {2*math.pi:.6f}   而弧長 = 8(沒有 pi!)")

# 畫擺線與生成圓
ts = np.linspace(0, 2*np.pi, 400)
plt.figure(figsize=(9, 3.2))
plt.plot(ts - np.sin(ts), 1 - np.cos(ts), lw=2, label='cycloid')
for t0 in [np.pi/2, np.pi, 3*np.pi/2]:
    c = np.linspace(0, 2*np.pi, 100)
    plt.plot(t0 + np.cos(c), 1 + np.sin(c), 'C1', lw=0.6)
    plt.plot([t0], [1], 'C1o', ms=3)
    plt.plot([t0 - np.sin(t0)], [1 - np.cos(t0)], 'C3o', ms=6)
plt.axhline(0, color='k', lw=0.8)
plt.gca().set_aspect('equal'); plt.legend()
plt.title('A rolling circle traces the cycloid')
plt.show()

In [ ]:
# TODO 學生練習:半徑改成 r=2(x = 2(t-sin t), y = 2(1-cos t))
# 弧長和面積各變成幾倍?驗證「弧長 8r、面積 3*pi*r^2」